In [3]:
import surprise
import pandas as pd
import numpy as np


import os
os.chdir(r'E:\PGCP-AI\PML\Cases\SURPRISE\Amazon Movie Ratings') 

In [15]:

df = pd.read_csv("Amazon.csv")

ratings = pd.melt(df, id_vars='user_id', 
                  var_name="item_id",value_name="rating")

ratings = ratings[ratings['rating'].notna()]


ratings

,user_id,item_id,rating
0,A3R5OBKS7OM2IR,Movie1,5.0
4848,A3R5OBKS7OM2IR,Movie2,5.0
9697,AH3QC2PC1VTGP,Movie3,2.0
14546,A3LKP6WPMP9UKX,Movie4,5.0
14547,AVIY68KEPQ5ZD,Movie4,5.0
...,...,...,...
998683,A1IMQ9WMFYKWH5,Movie206,5.0
998684,A1KLIKPUF5E88I,Movie206,5.0
998685,A5HG6WFZLO10D,Movie206,5.0
998686,A3UU690TWXCG1X,Movie206,5.0


In [16]:
lowest_rating = ratings['rating'].min()
highest_rating = ratings['rating'].max()
lowest_rating,highest_rating

(np.float64(1.0), np.float64(5.0))

In [6]:
reader =surprise.Reader(rating_scale=(lowest_rating, highest_rating))

data = surprise.Dataset.load_from_df(ratings,reader)
type(data)

surprise.dataset.DatasetAutoFolds

In [7]:
similarity_options = {'name':'cosine', 'user_based':True}
algo = surprise.KNNBasic(sim_options=similarity_options)
output = algo.fit(data.build_full_trainset())
# calculate the expected ratings for all the users

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [8]:
from surprise.model_selection import GridSearchCV
from surprise.model_selection.split import KFold 


# Finding best model for surprise.SVD

In [23]:
param_grid = {
    'n_epochs': [50],
    'n_factors':[100,50],
    'lr_all': np.linspace(0.001,1,5),
    'reg_all': np.linspace(0.01, 0.8, 5)
}
kfold = KFold(n_splits = 5, random_state = 26, shuffle = True)

gs = GridSearchCV(surprise.SVD , param_grid , measures = ['rmse','mae'], joblib_verbose = 3, cv = kfold , n_jobs = -1)
gs.fit(data)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  96 tasks      | elapsed:    3.5s
[Parallel(n_jobs=-1)]: Done 250 out of 250 | elapsed:    5.8s finished


In [24]:
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])

1.1597984189164987
{'n_epochs': 50, 'n_factors': 50, 'lr_all': np.float64(0.001), 'reg_all': np.float64(0.01)}


In [25]:
algo = gs.best_estimator['rmse']
algo.fit(data.build_full_trainset())

# Finding best model for surprise.KNNBasic

In [42]:
param_grid = {
    'k':[25,40,50,65,75,85],
    'min_k': [1,3,5],
    'sim_options': {
                    'name':['cosine'] , 
                    'user_based':[True,False]}
}
kfold = KFold(n_splits = 5, random_state = 26, shuffle = True)

gs = GridSearchCV(surprise.KNNBasic , param_grid , measures = ['rmse','mae'], joblib_verbose = 3, cv = kfold , n_jobs = -1)
gs.fit(data)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  96 tasks      | elapsed:    5.1s
[Parallel(n_jobs=-1)]: Done 180 out of 180 | elapsed:    8.6s finished


In [43]:
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])

1.1979005903278601
{'k': 25, 'min_k': 3, 'sim_options': {'name': 'cosine', 'user_based': True}}


In [44]:
algo = gs.best_estimator['rmse']
algo.fit(data.build_full_trainset())

Computing the cosine similarity matrix...
Done computing similarity matrix.


# Finding best model for surprise.KNNWithZScore

In [39]:
param_grid = {
    'k': [25, 40, 50, 65, 75, 85], 
    'sim_options': {
        'name': ['cosine', 'pearson', 'msd'], 
        'user_based': [False, True]
    }
}
kfold = KFold(n_splits = 5, random_state = 26, shuffle = True)
gs  = GridSearchCV(surprise.KNNWithZScore, param_grid, measures=['rmse','mae'], cv = kfold) 
gs.fit(data)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing

In [40]:
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])

1.1991484324293702
{'k': 25, 'sim_options': {'name': 'pearson', 'user_based': False}}


In [41]:
algo = gs.best_estimator['rmse'] 
algo.fit(data.build_full_trainset())

Computing the pearson similarity matrix...
Done computing similarity matrix.
